# 11｜Final Findings & Project Summary

這一本是整個專案的最後整理。

我不會在 11 再訓練新模型，也不會重新定義 SPC、Anomaly 或 OOD 規則。前面 02～10 已經完成的分析，我只做三件事：

1. **確認 artifacts 沒有接錯版本。**
2. **把真正重要的結果整理成同一條故事線。**
3. **把每個結果「可以說到哪裡、不能說到哪裡」一起寫清楚。**

所以 11 比較像 Final Report，而不是另一個新的 Modeling Notebook。

## 1. 讀入 02～10 的正式 artifacts

先把 Final Report 會用到的資料一次讀進來。

這裡不做 fallback，也不會在缺檔時偷偷重算。如果少任何一個必要 artifact，我寧願直接停掉，因為 Final Report 最怕的就是把不同版本的結果混在一起。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 200)


def find_project_root(start=None):
    # Walk upward until data/processed is found.
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError(
        "找不到專案根目錄：從目前工作目錄往上都沒有 data/processed。"
    )


PROJECT_ROOT = find_project_root()
PROCESSED_DATA_DIR = (PROJECT_ROOT / "data" / "processed").resolve()

paths = {
    # 02 / 03
    "quality": PROCESSED_DATA_DIR / "wafer_quality_summary.csv",
    "lot_quality": PROCESSED_DATA_DIR / "lot_quality_summary.csv",
    "radial": PROCESSED_DATA_DIR / "radial_summary.csv",
    "lot_sequence": PROCESSED_DATA_DIR / "lot_sequence_summary.csv",
    "process_summary": PROCESSED_DATA_DIR / "process_wafer_summary.csv",
    "process_sequence_summary": PROCESSED_DATA_DIR / "process_sequence_summary.csv",

    # 04 / 05 / 06
    "mean_candidates": PROCESSED_DATA_DIR / "mean_etch_exploratory_candidates.csv",
    "cv_candidates": PROCESSED_DATA_DIR / "cv_exploratory_candidates.csv",
    "spc_feature": PROCESSED_DATA_DIR / "spc_feature_summary.csv",
    "spc_wafer": PROCESSED_DATA_DIR / "spc_wafer_flag_summary.csv",
    "feature_sequence_comparison": PROCESSED_DATA_DIR / "06_feature_sequence_comparison.csv",
    "raw_pca_var": PROCESSED_DATA_DIR / "06_raw_pca_variance.csv",
    "residual_pca_var": PROCESSED_DATA_DIR / "06_residual_pca_variance.csv",

    # 08 / 09
    "baseline": PROCESSED_DATA_DIR / "08_baseline_metrics.csv",
    "outer": PROCESSED_DATA_DIR / "08_nested_outer_fold_results.csv",
    "oof": PROCESSED_DATA_DIR / "08_nested_oof_predictions.csv",
    "config": PROCESSED_DATA_DIR / "08_final_model_config.json",
    "coef_stability": PROCESSED_DATA_DIR / "09_coefficient_stability.csv",
    "sensor_coef": PROCESSED_DATA_DIR / "09_sensor_coefficient_summary.csv",
    "sensor_perm": PROCESSED_DATA_DIR / "09_sensor_grouped_permutation_importance.csv",
    "error_by_lot": PROCESSED_DATA_DIR / "09_nested_error_by_lot.csv",

    # 10
    "residual_anomaly": PROCESSED_DATA_DIR / "10_residual_anomaly_scores.csv",
    "ood": PROCESSED_DATA_DIR / "10_cross_lot_ood_scores.csv",
    "ood_thresholds": PROCESSED_DATA_DIR / "10_ood_fold_thresholds.csv",
    "tri": PROCESSED_DATA_DIR / "10_anomaly_triangulation.csv",
    "lot": PROCESSED_DATA_DIR / "10_anomaly_by_lot.csv",
    "association": PROCESSED_DATA_DIR / "10_process_vs_model_error_association.csv",
    "priority": PROCESSED_DATA_DIR / "10_priority_cases.csv",
    "methodology": PROCESSED_DATA_DIR / "10_anomaly_methodology.json",
}

missing = [p for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "11 缺少必要 upstream artifacts：\n" + "\n".join(str(p) for p in missing)
    )

quality = pd.read_csv(paths["quality"])
lot_quality = pd.read_csv(paths["lot_quality"])
radial = pd.read_csv(paths["radial"])
lot_sequence = pd.read_csv(paths["lot_sequence"])
process_summary = pd.read_csv(paths["process_summary"])
process_sequence_summary = pd.read_csv(paths["process_sequence_summary"])

mean_candidates = pd.read_csv(paths["mean_candidates"])
cv_candidates = pd.read_csv(paths["cv_candidates"])
spc_feature = pd.read_csv(paths["spc_feature"])
spc_wafer = pd.read_csv(paths["spc_wafer"])
feature_sequence_comparison = pd.read_csv(paths["feature_sequence_comparison"])
raw_pca_var = pd.read_csv(paths["raw_pca_var"])
residual_pca_var = pd.read_csv(paths["residual_pca_var"])

baseline = pd.read_csv(paths["baseline"])
outer = pd.read_csv(paths["outer"])
nested_oof = pd.read_csv(paths["oof"])
coef_stability = pd.read_csv(paths["coef_stability"])
sensor_coef = pd.read_csv(paths["sensor_coef"])
sensor_perm = pd.read_csv(paths["sensor_perm"])
error_by_lot = pd.read_csv(paths["error_by_lot"])

residual_anomaly = pd.read_csv(paths["residual_anomaly"])
cross_lot_ood = pd.read_csv(paths["ood"])
ood_fold_thresholds = pd.read_csv(paths["ood_thresholds"])
tri = pd.read_csv(paths["tri"])
lot_summary = pd.read_csv(paths["lot"])
association = pd.read_csv(paths["association"])
priority_cases = pd.read_csv(paths["priority"])

with open(paths["config"], "r", encoding="utf-8") as f:
    final_config = json.load(f)

with open(paths["methodology"], "r", encoding="utf-8") as f:
    anomaly_methodology = json.load(f)

print("Final Report artifacts 載入完成")
print("Process wafers：", len(tri))
print("Quality-labeled / OOF wafers：", len(nested_oof))
print("Lots：", tri["lot_number"].nunique())

Final Report artifacts 載入完成
Process wafers： 96
Quality-labeled / OOF wafers： 88
Lots： 10


### 這個結果代表什麼？

這一格確認 Final Report 需要的 artifacts 都可以正常讀取。

目前整個專案的母體是：

- **96 片**有 Process Data
- **88 片**有 Quality label，也有 Nested OOF prediction
- **10 個 Lots**

這裡只是先確認 11 接到的是前面真正輸出的資料。11 不會另外建立一套 threshold、model 或 candidate rule，避免最後統整時又出現另一個版本。

## 2. Cross-Artifact Validation：先確認 02～10 真的是同一條資料流程

Final Report 最重要的不是表很多，而是前後不能互相矛盾。

所以我先驗幾條 lineage：

- Quality wafers 和 08 Nested OOF 必須是同一批 labeled wafers。
- 08 的 `nested_error` 必須真的等於 prediction − actual。
- 09 per-Lot error 必須能從 08 OOF 重新算回來。
- 10 的 Lot-level VM RMSE 必須和 08 OOF 一致。
- 06 residual PCA 和 10 residual anomaly 必須使用同一個 residual universe。
- 10 Pure Cross-Lot OOD 必須確認：
  - 只使用 07 process engineered features。
  - 不使用 Quality target。
  - 不使用 08 error。
  - 不使用 09 importance。
  - Inner calibration 和 final outer detector 使用同一套 **outer-training-only fixed feature universe**。

In [2]:
def assert_unique_key(df, name):
    if "experiment_key" not in df.columns:
        raise ValueError(f"{name} 缺少 experiment_key")
    if df["experiment_key"].duplicated().any():
        raise ValueError(f"{name} experiment_key 有重複")


for name, df in {
    "quality": quality,
    "nested_oof": nested_oof,
    "residual_anomaly": residual_anomaly,
    "cross_lot_ood": cross_lot_ood,
    "triangulation": tri,
}.items():
    assert_unique_key(df, name)

# Identity lineage
if set(quality["experiment_key"]) != set(nested_oof["experiment_key"]):
    raise ValueError("Quality / Nested OOF wafer identity 不一致")

if not set(quality["experiment_key"]).issubset(set(tri["experiment_key"])):
    raise ValueError("Labeled wafers 不是 Process population 的子集")

if set(residual_anomaly["experiment_key"]) != set(tri["experiment_key"]):
    raise ValueError("Residual anomaly population 與 triangulation 不一致")

if set(cross_lot_ood["experiment_key"]) != set(tri["experiment_key"]):
    raise ValueError("Cross-Lot OOD population 與 triangulation 不一致")

# 08 prediction error formula
calc_error = (
    nested_oof["nested_process_pred"].to_numpy(float)
    - nested_oof["actual_mean_si_etch"].to_numpy(float)
)

if not np.allclose(calc_error, nested_oof["nested_error"].to_numpy(float), atol=1e-12, rtol=0):
    raise ValueError("08 nested_error formula 不一致")

if not np.allclose(
    np.abs(calc_error),
    nested_oof["abs_nested_error"].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError("08 abs_nested_error formula 不一致")

# Rebuild per-Lot VM error from 08 OOF and compare with 09 / 10.
recalc_lot = (
    nested_oof.groupby("lot_number", as_index=False)
    .agg(
        n_wafers=("experiment_key", "size"),
        rmse=("nested_error", lambda s: float(np.sqrt(np.mean(np.asarray(s, float) ** 2)))),
        mae=("abs_nested_error", "mean"),
        max_abs_error=("abs_nested_error", "max"),
    )
    .sort_values("lot_number")
    .reset_index(drop=True)
)

err_check = error_by_lot.sort_values("lot_number").reset_index(drop=True)
for col in ["n_wafers", "rmse", "mae", "max_abs_error"]:
    if not np.allclose(
        recalc_lot[col].to_numpy(float),
        err_check[col].to_numpy(float),
        atol=1e-12,
        rtol=0,
    ):
        raise ValueError(f"08 → 09 Lot error lineage 不一致：{col}")

lot_vm = lot_summary[["lot_number", "vm_rmse", "vm_max_abs_error"]].sort_values("lot_number")
lot_vm = lot_vm.reset_index(drop=True)

if not np.allclose(lot_vm["vm_rmse"], recalc_lot["rmse"], atol=1e-12, rtol=0):
    raise ValueError("08 → 10 Lot VM RMSE lineage 不一致")

if not np.allclose(lot_vm["vm_max_abs_error"], recalc_lot["max_abs_error"], atol=1e-12, rtol=0):
    raise ValueError("08 → 10 Lot VM max error lineage 不一致")

# 06 -> 10 residual lineage
n_resid_06 = len(residual_pca_var)
n_resid_10 = int(anomaly_methodology["residual_feature_count"])
if n_resid_06 != n_resid_10:
    raise ValueError(f"06 / 10 residual feature count 不一致：{n_resid_06} vs {n_resid_10}")

pca90_06 = int(
    residual_pca_var.loc[
        residual_pca_var["cumulative_variance_ratio"] >= 0.90,
        "component",
    ].iloc[0]
)
pca90_10 = int(anomaly_methodology["residual_pca_components_90pct"])

if pca90_06 != pca90_10:
    raise ValueError(f"06 / 10 residual PCA 90% components 不一致：{pca90_06} vs {pca90_10}")

# 10 Pure OOD source separation
if anomaly_methodology.get("pure_cross_lot_ood_uses_quality_target") is not False:
    raise ValueError("Pure OOD 不應使用 Quality target")

if anomaly_methodology.get("pure_cross_lot_ood_uses_notebook08_error") is not False:
    raise ValueError("Pure OOD 不應使用 08 prediction error")

if anomaly_methodology.get("pure_cross_lot_ood_uses_notebook09_importance") is not False:
    raise ValueError("Pure OOD 不應使用 09 importance")

if anomaly_methodology.get("pure_cross_lot_ood_fixed_outer_feature_universe") is not True:
    raise ValueError("Pure OOD 尚未使用 fixed outer feature universe")

if not ood_fold_thresholds["calibration_uses_fixed_outer_feature_universe"].astype(bool).all():
    raise ValueError("OOD fold audit 顯示 calibration feature universe 不一致")

if len(ood_fold_thresholds) != tri["lot_number"].nunique():
    raise ValueError("OOD fold threshold rows 與 Lot 數不一致")

print("02 / 08 labeled wafer identity：PASS")
print("08 prediction error formula：PASS")
print("08 → 09 → 10 VM error lineage：PASS")
print("06 → 10 residual PCA lineage：PASS")
print("10 Pure OOD source separation：PASS")
print("10 fixed outer feature universe：PASS")

02 / 08 labeled wafer identity：PASS
08 prediction error formula：PASS
08 → 09 → 10 VM error lineage：PASS
06 → 10 residual PCA lineage：PASS
10 Pure OOD source separation：PASS
10 fixed outer feature universe：PASS


### 這個結果代表什麼？

這一格的六項檢查全部都是 `PASS`。

也就是說：

- 02 的 Quality labels 和 08 的 OOF predictions 是同一批 wafers。
- 08 → 09 → 10 的 prediction error 沒有在中間被改掉。
- 06 的 residual PCA universe 和 10 residual anomaly 使用的是同一套資料定義。
- 10 的 Pure Cross-Lot OOD 沒有把 Quality、08 model error 或 09 supervised importance 偷混進去。
- OOD calibration 也已經是修正後的 **fixed outer feature universe** 版本。

所以後面 11 的數字可以放在同一份 Final Report 裡，不是在拼接不同版本的結果。

## 3. Project Headline Metrics：先把整個專案最重要的數字放在一起

這一格只整理最核心的 headline，不做新的模型。

我會同時放：

- Quality 的基本結果
- Sequence structure
- Virtual Metrology performance
- SPC / Residual / OOD 的 review counts

這樣後面每一節再分開解釋。

In [3]:
y = nested_oof["actual_mean_si_etch"].to_numpy(float)
pred = nested_oof["nested_process_pred"].to_numpy(float)

vm_rmse = float(np.sqrt(np.mean((pred - y) ** 2)))
vm_mae = float(np.mean(np.abs(pred - y)))
vm_r2 = float(r2_score(y, pred))

mean_base = float(
    baseline.loc[baseline["model"] == "Training-Mean Baseline", "rmse"].iloc[0]
)
seq_base = float(
    baseline.loc[baseline["model"] == "Sequence-Only Linear Baseline", "rmse"].iloc[0]
)

inner_mask = radial["radius_region"] != ">=75 mm"
inner_mean = float(
    np.average(
        radial.loc[inner_mask, "mean_si_etch"],
        weights=radial.loc[inner_mask, "n_points"],
    )
)
outer_mean = float(
    radial.loc[radial["radius_region"] == ">=75 mm", "mean_si_etch"].iloc[0]
)

res_count = int(residual_anomaly["residual_review_candidate"].sum())
ood_count = int(cross_lot_ood["cross_lot_ood_candidate"].sum())

candidate_overlap = (
    residual_anomaly[["experiment_key", "residual_review_candidate"]]
    .merge(
        cross_lot_ood[["experiment_key", "cross_lot_ood_candidate"]],
        on="experiment_key",
        how="inner",
        validate="one_to_one",
    )
)
overlap_count = int(
    (
        candidate_overlap["residual_review_candidate"]
        & candidate_overlap["cross_lot_ood_candidate"]
    ).sum()
)

headline = pd.DataFrame([
    {"metric": "Process wafers", "value": len(tri), "unit": "wafers"},
    {"metric": "Quality-labeled wafers", "value": len(nested_oof), "unit": "wafers"},
    {"metric": "Lots", "value": tri["lot_number"].nunique(), "unit": "Lots"},
    {"metric": "Mean Si Etch", "value": quality["mean_si_etch"].mean(), "unit": "µm"},
    {"metric": "Mean CV", "value": quality["cv_si_etch_pct"].mean(), "unit": "%"},
    {"metric": "Outer - inner radial mean", "value": outer_mean - inner_mean, "unit": "µm"},
    {"metric": "Nested OOF RMSE", "value": vm_rmse, "unit": "µm"},
    {"metric": "Nested OOF MAE", "value": vm_mae, "unit": "µm"},
    {"metric": "Nested OOF R²", "value": vm_r2, "unit": ""},
    {"metric": "Sequence-only baseline RMSE", "value": seq_base, "unit": "µm"},
    {"metric": "Training-mean baseline RMSE", "value": mean_base, "unit": "µm"},
    {"metric": "Residual review candidates", "value": res_count, "unit": "wafers"},
    {"metric": "Cross-Lot OOD candidates", "value": ood_count, "unit": "wafers"},
    {"metric": "Residual + OOD overlap", "value": overlap_count, "unit": "wafers"},
])

display(headline.round(5))

,metric,value,unit
0,Process wafers,96.00000,wafers
1,Quality-labeled wafers,88.00000,wafers
2,Lots,10.00000,Lots
3,Mean Si Etch,43.66812,µm
4,Mean CV,8.21143,%
5,Outer - inner radial mean,5.03895,µm
6,Nested OOF RMSE,0.12963,µm
7,Nested OOF MAE,0.09723,µm
8,Nested OOF R²,0.89477,
9,Sequence-only baseline RMSE,0.22093,µm


### 這個結果代表什麼？

先看整體資料規模：目前有 **96 片 Process wafers、88 片 Quality-labeled wafers、10 個 Lots**。

Quality 方面，平均 Mean Si Etch 是 **43.66812 µm**，平均 CV 是 **8.21143%**。把半徑 `>=75 mm` 的外圈和內部量測區做 pooled comparison，外圈平均比內部高 **5.03895 µm**，表示 wafer 上有很明顯的 radial spatial structure。

Virtual Metrology 的正式 Nested OOF 成績是 **RMSE = 0.12963 µm、MAE = 0.09723 µm、R² = 0.89477**。

10 的 review 結果則是 **3 片 residual candidates、5 片 Cross-Lot OOD candidates**，其中真正同時被兩種方法標記的是 **2 片**。

所以這個專案不是只有一個模型分數，而是同時有 Quality、Process structure、prediction、SPC 和 OOD 幾個不同層次的結果。

## 4. Quality 與 Sequence Structure：先確認資料本身有哪些穩定結構

在做 Process–Quality correlation 或模型以前，先把最重要的資料結構整理出來：

- Quality 是否有 Lot 內 run-order trend？
- Process features 是否也跟 wafer sequence 有關？
- 扣掉線性 sequence trend 後，這種 association 有沒有真的下降？
- Main Active Window 的時間範圍是否穩定？

這一節的目的不是證明因果，而是說明為什麼後面的分析不能把 96 片 wafer 當成完全獨立、同分布的資料。

In [4]:
quality_sequence_summary = pd.DataFrame([
    {
        "metric": "Mean Si Etch average",
        "value": float(quality["mean_si_etch"].mean()),
        "note": "88 labeled wafers",
    },
    {
        "metric": "CV average (%)",
        "value": float(quality["cv_si_etch_pct"].mean()),
        "note": "88 labeled wafers",
    },
    {
        "metric": "Lots with negative Mean Si Etch slope",
        "value": int((lot_sequence["mean_si_etch_slope_per_wafer"] < 0).sum()),
        "note": f"out of {len(lot_sequence)} Lots",
    },
    {
        "metric": "Lots with negative CV slope",
        "value": int((lot_sequence["cv_slope_per_wafer"] < 0).sum()),
        "note": f"out of {len(lot_sequence)} Lots",
    },
    {
        "metric": "Main Active Window mean duration (s)",
        "value": float(process_summary["main_active_duration"].mean()),
        "note": "03 data-driven 80% SourceRFLoadPower heuristic",
    },
    {
        "metric": "Main Active Window duration SD (s)",
        "value": float(process_summary["main_active_duration"].std()),
        "note": "96 process wafers",
    },
    {
        "metric": "Median |within-Lot Spearman| before detrending",
        "value": float(feature_sequence_comparison["raw_median_abs_within_lot_spearman"].median()),
        "note": "24 comparable process features",
    },
    {
        "metric": "Median |within-Lot Spearman| after detrending",
        "value": float(feature_sequence_comparison["residual_median_abs_within_lot_spearman"].median()),
        "note": "same 24 process features",
    },
])

display(quality_sequence_summary.round(5))

top_sequence_features = (
    process_sequence_summary
    .sort_values("median_abs_spearman", ascending=False)
    .head(8)
    [["feature", "dominant_direction", "direction_consistency_pct", "median_spearman", "median_abs_spearman"]]
    .copy()
)
top_sequence_features["feature"] = top_sequence_features["feature"].str.replace(
    "Stat3_Etch_MV_", "", regex=False
)

print("\nSequence structure 最明顯的 process features：")
display(top_sequence_features.round(4))

n_decreased = int(
    (
        feature_sequence_comparison["residual_median_abs_within_lot_spearman"]
        < feature_sequence_comparison["raw_median_abs_within_lot_spearman"]
    ).sum()
)
print(
    f"Sequence detrending 後 |Spearman| 下降："
    f"{n_decreased} / {len(feature_sequence_comparison)} features"
)

,metric,value,note
0,Mean Si Etch average,43.66812,88 labeled wafers
1,CV average (%),8.21143,88 labeled wafers
2,Lots with negative Mean Si Etch slope,10.00000,out of 10 Lots
3,Lots with negative CV slope,9.00000,out of 10 Lots
4,Main Active Window mean duration (s),597.54208,03 data-driven 80% SourceRFLoadPower heuristic
5,Main Active Window duration SD (s),2.19824,96 process wafers
6,Median |within-Lot Spearman| before detrending,0.38788,24 comparable process features
7,Median |within-Lot Spearman| after detrending,0.08485,same 24 process features



Sequence structure 最明顯的 process features：


,feature,dominant_direction,direction_consistency_pct,median_spearman,median_abs_spearman
0,PlatenRFTuningCapacitor,Increasing,100.0,0.9576,0.9576
1,SourceRFPeakToPeak,Decreasing,100.0,-0.8061,0.8061
2,ForeLinePressure,Decreasing,100.0,-0.7758,0.7758
3,PlatenRFPeakToPeak,Increasing,100.0,0.7697,0.7697
4,SourceRFLoadPower,Decreasing,100.0,-0.6727,0.6727
8,Gas7Flow,Increasing,80.0,0.5576,0.5576
19,Gas2Flow,Decreasing,60.0,0.0000,0.5152
15,Heater3Temp,Increasing,70.0,0.4182,0.5091


Sequence detrending 後 |Spearman| 下降：24 / 24 features


### 這個結果代表什麼？

Quality 本身就有很明顯的 run-order structure：

- Mean Si Etch 的 Lot 內斜率在 **10 / 10 Lots** 都是負的。
- CV 的 Lot 內斜率在 **9 / 10 Lots** 是負的。

Process 端也有類似現象。24 個可以前後比較的 process features，raw median `|within-Lot Spearman|` 是 **0.38788**；扣掉 Lot-specific linear sequence trend 後降到 **0.08485**，而且 **24 / 24** 個 features 都下降。

這表示 wafer sequence 不是一個可以忽略的小細節。後面如果直接把所有 wafers 混在一起做 correlation，很容易把 run-order structure 誤認成 Process–Quality relation。

另外 Main Active Window 平均約 **597.54 秒**，標準差約 **2.20 秒**。這個 window 是 03 用 `SourceRFLoadPower >= wafer max 的 80%` 找到的 data-driven heuristic，所以我會把它叫作一致的分析區段，不會說它是官方 recipe boundary。

## 5. Process–Quality：扣掉 Lot 內線性 sequence trend 後，還留下哪些候選？

04 的正式 candidate rule 已經在上游完成，11 不再重新篩選。

這裡只整理兩個 target：

- Mean Si Etch
- CV Si Etch

這些候選可以拿來安排 DOE、equipment log review 或 recipe review 的優先順序，但不能直接寫成 causal driver。

In [5]:
def short_sensor(series):
    return series.astype(str).str.replace("Stat3_Etch_MV_", "", regex=False)

mean_final = mean_candidates.copy()
mean_final.insert(0, "sensor", short_sensor(mean_final["feature"]))
mean_final["target"] = "Mean Si Etch"

cv_final = cv_candidates.copy()
cv_final.insert(0, "sensor", short_sensor(cv_final["feature"]))
cv_final["target"] = "CV Si Etch"

process_quality_summary = pd.concat([mean_final, cv_final], ignore_index=True)

print("Mean Si Etch candidates：")
display(
    mean_final[
        ["sensor", "sequence_adjusted_pearson", "sequence_adjusted_spearman", "n_valid_lots"]
    ].round(4)
)

print("CV Si Etch candidates：")
display(
    cv_final[
        ["sensor", "sequence_adjusted_pearson", "sequence_adjusted_spearman", "n_valid_lots"]
    ].round(4)
)

top_mean = mean_final.loc[mean_final["abs_sequence_adjusted_pearson"].idxmax()]
top_cv = cv_final.loc[cv_final["abs_sequence_adjusted_pearson"].idxmax()]

Mean Si Etch candidates：


,sensor,sequence_adjusted_pearson,sequence_adjusted_spearman,n_valid_lots
0,PlatenRFLoadPower,-0.6090,-0.4855,10
1,HeliumBPPressure,-0.3999,-0.4042,10
2,Gas1Flow,0.3298,0.1753,10
3,PlatenRFPeakToPeak,-0.3290,-0.3128,10
4,Pressure,-0.3079,-0.2382,10


CV Si Etch candidates：


,sensor,sequence_adjusted_pearson,sequence_adjusted_spearman,n_valid_lots
0,Heater4Temp,0.5513,0.4654,10
1,moriInnerCurrent,-0.5008,-0.5020,10
2,Gas7Flow,-0.4201,-0.5653,10
3,PlatenRFTuningCapacitor,-0.4107,-0.6259,10
4,PlatenRFPeakToPeak,-0.4088,-0.5157,10
5,Gas2Flow,0.3277,0.2363,10
6,Heater2Temp,0.3158,0.2005,10


### 這個結果代表什麼？

扣掉 Lot 內線性 wafer-sequence trend 後：

- Mean Si Etch 還有 **5 個** exploratory candidates。
- CV Si Etch 還有 **7 個** exploratory candidates。

Mean Si Etch 這組裡，`PlatenRFLoadPower` 的 sequence-adjusted Pearson 最明顯，**r = -0.6090**。

CV 這組裡，`Heater4Temp` 最明顯，**r = +0.5513**。

這表示 sequence effect 雖然很重要，但扣掉線性 trend 之後，Process–Quality association 並沒有全部消失。

不過這些數字仍然是 observational association。比較合理的用法是：「哪些 sensors 值得優先做 DOE 或查設備紀錄」，而不是直接說「調這個 sensor 就會讓 etch quality 改變」。

## 6. SPC 與 Multivariate Structure：哪些 wafer / feature 值得先回查？

05 是 retrospective Phase-I SPC；06 則用 PCA 看多變量結構。

這裡要特別分清楚：

- I-chart flag：某片 wafer 的 residual level 比較極端。
- MR flag：前一片到這一片的變化比較大。
- PCA：只是幫忙描述多變量 variation，不是製程物理模型。

所以這些訊號適合當 investigation trigger，不是 production defect label。

In [6]:
spc_summary = pd.DataFrame([
    {
        "metric": "I-chart feature-wafer flags",
        "value": int(spc_wafer["n_flagged_features"].sum()),
    },
    {
        "metric": "Unique wafers with I-chart flag",
        "value": int((spc_wafer["n_flagged_features"] > 0).sum()),
    },
    {
        "metric": "MR feature-wafer flags",
        "value": int(spc_wafer["n_mr_flagged_features"].sum()),
    },
    {
        "metric": "Unique wafers with MR flag",
        "value": int((spc_wafer["n_mr_flagged_features"] > 0).sum()),
    },
    {
        "metric": "Raw PCA variance explained by PC1-PC3",
        "value": float(raw_pca_var["explained_variance_ratio"].head(3).sum()),
    },
    {
        "metric": "Raw PCA components for >=90%",
        "value": int(raw_pca_var.loc[raw_pca_var["cumulative_variance_ratio"] >= 0.90, "component"].iloc[0]),
    },
    {
        "metric": "Residual PCA components for >=90%",
        "value": int(residual_pca_var.loc[residual_pca_var["cumulative_variance_ratio"] >= 0.90, "component"].iloc[0]),
    },
])

display(spc_summary.round(5))

print("\nI-chart violations 最多的 features：")
top_spc_features = (
    spc_feature.sort_values("n_i_violations", ascending=False)
    .head(8)
    [["feature", "n_i_violations", "i_violation_rate_pct", "n_mr_violations", "mr_violation_rate_pct"]]
    .copy()
)
top_spc_features["feature"] = top_spc_features["feature"].str.replace(
    "Stat3_Etch_MV_", "", regex=False
)
display(top_spc_features.round(4))

,metric,value
0,I-chart feature-wafer flags,46.00000
1,Unique wafers with I-chart flag,19.00000
2,MR feature-wafer flags,47.00000
3,Unique wafers with MR flag,23.00000
4,Raw PCA variance explained by PC1-PC3,0.48884
5,Raw PCA components for >=90%,12.00000
6,Residual PCA components for >=90%,12.00000



I-chart violations 最多的 features：


,feature,n_i_violations,i_violation_rate_pct,n_mr_violations,mr_violation_rate_pct
0,Gas7Flow,8,8.3333,4,4.6512
1,PlatenRFTuningCapacitor,7,7.2917,0,0.0000
2,Heater4Temp,5,5.2083,2,2.3256
3,Gas1Flow,5,5.2083,8,9.3023
4,Heater2Temp,3,3.1250,2,2.3256
5,SourceRFLoadPower,3,3.1250,5,5.8140
6,SourceRF2TuningCapacitor,3,5.0000,4,7.4074
7,Gas2Flow,2,2.0833,2,2.3256


### 這個結果代表什麼？

Phase-I SPC 一共出現 **46 個 I-chart feature-wafer flags**，分布在 **19 片 wafers**；MR 則有 **47 個 feature-wafer flags**，分布在 **23 片 wafers**。

I-chart violations 最多的是 `Gas7Flow`，共有 **8 片**被 flag。

Multivariate 方面，raw PCA 前 3 個 PCs 一共解釋 **48.88%** variation，而累積到 90% 需要 **12 個 PCs**。Residual PCA 同樣需要 **12 個 PCs** 才到 90%。

所以這批 process variation 不是一兩個方向就能完全概括。SPC 可以指出值得回查的 wafer / feature，但目前這些 limits 是 retrospective Phase-I provisional limits，不能直接當 production control limits。

## 7. Virtual Metrology：正式成績只看 Nested Leave-One-Lot-Out

08 的 full-data final model 是 prototype configuration；真正可以拿來談 generalization 的成績，仍然是 Nested Leave-One-Lot-Out OOF。

這裡同時保留兩個簡單 baseline：

- Training-Mean baseline
- Sequence-Only Linear baseline

這樣可以回答 process-feature model 到底有沒有比很簡單的做法更好。

In [7]:
print("Full-data prototype configuration：")
print(
    final_config["feature_set"],
    "+",
    final_config["model"],
    final_config["params"],
)
print(
    "Input predictors / variance filter 後保留：",
    final_config["n_input_predictors"],
    "/",
    final_config["n_predictors_after_variance_filter"],
)

vm_summary = pd.DataFrame([
    {"metric": "Nested OOF RMSE (µm)", "value": vm_rmse},
    {"metric": "Nested OOF MAE (µm)", "value": vm_mae},
    {"metric": "Nested OOF R²", "value": vm_r2},
    {"metric": "Sequence-only RMSE (µm)", "value": seq_base},
    {"metric": "Training-mean RMSE (µm)", "value": mean_base},
    {"metric": "RMSE reduction vs sequence-only", "value": (seq_base - vm_rmse) / seq_base},
    {"metric": "RMSE reduction vs training-mean", "value": (mean_base - vm_rmse) / mean_base},
])
display(vm_summary.round(5))

print("\nPer-Lot held-out performance：")
display(recalc_lot.sort_values("rmse", ascending=False).round(5))

print("\nOuter-fold selected feature sets：")
display(outer["feature_set"].value_counts().rename_axis("feature_set").reset_index(name="folds"))

print("\nOuter-fold selected model families：")
display(outer["model"].value_counts().rename_axis("model").reset_index(name="folds"))

Full-data prototype configuration：
Extended + ElasticNet {'alpha': 0.01, 'l1_ratio': 0.1}
Input predictors / variance filter 後保留： 109 / 98


,metric,value
0,Nested OOF RMSE (µm),0.12963
1,Nested OOF MAE (µm),0.09723
2,Nested OOF R²,0.89477
3,Sequence-only RMSE (µm),0.22093
4,Training-mean RMSE (µm),0.40437
5,RMSE reduction vs sequence-only,0.41324
6,RMSE reduction vs training-mean,0.67942



Per-Lot held-out performance：


,lot_number,n_wafers,rmse,mae,max_abs_error
7,8,10,0.22384,0.14756,0.57555
2,3,10,0.14304,0.12877,0.22756
4,5,10,0.13771,0.11333,0.27007
6,7,6,0.11837,0.09339,0.20857
5,6,10,0.11752,0.08241,0.27963
3,4,10,0.11432,0.09827,0.21846
1,2,10,0.10996,0.09629,0.16586
9,10,4,0.08315,0.07746,0.10814
8,9,9,0.06980,0.05825,0.12378
0,1,9,0.06516,0.05508,0.11583



Outer-fold selected feature sets：


,feature_set,folds
0,Extended,6
1,Base,4



Outer-fold selected model families：


,model,folds
0,ElasticNet,10


### 這個結果代表什麼？

正式 Nested OOF 成績是：

- RMSE = **0.12963 µm**
- MAE = **0.09723 µm**
- R² = **0.89477**

同一套 nested evaluation 下：

- Sequence-only baseline RMSE = **0.22093 µm**
- Training-mean baseline RMSE = **0.40437 µm**

所以 process-feature model 的 RMSE 分別低約 **41.32%** 和 **67.94%**。

10 個 outer folds 最後全部選到 **ElasticNet**；feature set 則有 **6 folds 選 Extended、4 folds 選 Base**。

這表示 process features 在目前 held-out-Lot evaluation 裡確實有預測價值。不過這還不是「先完全控制 sequence，再只量 process features 額外價值」的 ablation，所以不能把 baseline 差距全部解讀成 process features 的獨立增量效果。

另外每個 Lot 的 RMSE 差距很大，也代表 pooled RMSE 不能代替 group-level monitoring。

## 8. Explainability：模型最依賴哪些 sensor groups？

09 有兩種互補的 model-dependent evidence：

1. Nested outer models 的 grouped permutation sensitivity。
2. Final ElasticNet 的 standardized coefficient aggregation。

如果兩邊都指向同一個 sensor group，可以說模型對這組資訊的依賴比較一致；但仍然不能直接把它叫作 causal process driver。

In [8]:
top_perm = (
    sensor_perm
    .sort_values("mean_delta_rmse", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("Nested grouped permutation sensitivity：")
display(top_perm.round(5))

top_coef = (
    sensor_coef
    .sort_values("abs_coefficient_sum", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("\nFinal ElasticNet sensor coefficient aggregation：")
display(top_coef.round(5))

top_perm_sensor = str(top_perm.iloc[0]["sensor"])
top_coef_sensor = str(top_coef.iloc[0]["sensor_short"])
print("\nTop sensor agrees across both views：", top_perm_sensor == top_coef_sensor)

Nested grouped permutation sensitivity：


,sensor,n_repeats,mean_delta_rmse,std_delta_rmse,positive_repeat_fraction
0,PlatenRFTuningCapacitor,30,0.24996,0.02185,1.00000
1,MainActiveWindow,30,0.05900,0.01242,1.00000
2,ForeLinePressure,30,0.03235,0.00597,1.00000
3,SourceRFReflectedPower,30,0.02489,0.00461,1.00000
4,PlatenRFReflectedPower,30,0.01596,0.00366,1.00000
5,SourceRFLoadPower,30,0.01398,0.00440,1.00000
6,SourceRFPeakToPeak,30,0.00924,0.00344,1.00000
7,PlatenRFLoadCapacitor,30,0.00333,0.00187,0.96667
8,PlatenRFLoadPower,30,0.00200,0.00094,0.96667
9,Gas2Flow,30,0.00151,0.00131,0.83333



Final ElasticNet sensor coefficient aggregation：


,sensor_short,n_features,n_nonzero,abs_coefficient_sum,max_abs_coefficient
0,PlatenRFTuningCapacitor,4,3,0.26939,0.15959
1,PlatenRFReflectedPower,4,3,0.11761,0.09083
2,ForeLinePressure,4,4,0.10781,0.07588
3,MainActiveWindow,1,1,0.09182,0.09182
4,SourceRFReflectedPower,4,2,0.07620,0.05684
5,Heater2Temp,4,3,0.07293,0.05977
6,SourceRFPeakToPeak,4,2,0.05505,0.04177
7,PlatenRFPeakToPeak,4,3,0.05198,0.02665
8,SourceRFLoadPower,4,2,0.05103,0.04794
9,Gas7Flow,4,3,0.04582,0.02355



Top sensor agrees across both views： True


### 這個結果代表什麼？

Grouped permutation 裡最明顯的是 **`PlatenRFTuningCapacitor`**，平均 ΔRMSE = **+0.24996 µm**，明顯高於後面的 sensor groups。

Final ElasticNet 的 coefficient aggregation 第一名也是 `PlatenRFTuningCapacitor`，所以兩種方法在最前面的 sensor group 上是一致的。

我會把這個結果解讀成：**目前模型反覆使用這組資訊來做 prediction。**

但是它仍然是 predictive dependence。Sensor 之間有 correlation 時，importance 可能被共享或替代，所以這個排名不能直接變成「製程因果重要性排名」。

## 9. Residual Anomaly、Pure Cross-Lot OOD 與 Model Error：三件事分開看

10 已經把這幾個概念分開：

- **Residual anomaly**：看扣掉 Lot-specific linear sequence trend 後，還有沒有少見 residual pattern。
- **Pure Cross-Lot OOD**：只用 07 process engineered features，看 held-out Lot 相對其他 Lots 有多陌生。
- **VM-informed diagnostic**：可以用 09 supervised model sensitivity 做 post-hoc model context，但它不參與 Pure OOD candidate flag。

11 只讀 10 的正式輸出，不重新建立 detector。

In [9]:
res_candidates = (
    residual_anomaly.loc[
        residual_anomaly["residual_review_candidate"],
        ["experiment_key", "lot_number", "wafer_number", "residual_anomaly_consensus", "residual_driver_feature"],
    ]
    .sort_values(["lot_number", "wafer_number"])
    .reset_index(drop=True)
)

ood_candidates = (
    cross_lot_ood.loc[
        cross_lot_ood["cross_lot_ood_candidate"],
        ["experiment_key", "lot_number", "wafer_number", "cross_lot_ood_consensus", "ood_driver_feature"],
    ]
    .sort_values(["lot_number", "wafer_number"])
    .reset_index(drop=True)
)

overlap_keys = sorted(
    set(res_candidates["experiment_key"])
    & set(ood_candidates["experiment_key"])
)

print("Residual review candidates：", len(res_candidates))
display(res_candidates)

print("Cross-Lot OOD candidates：", len(ood_candidates))
display(ood_candidates)

print("Residual + OOD overlap：", len(overlap_keys), overlap_keys)

print("\nProcess score vs |OOF error|：")
display(association.round(5))

print("\nPriority cases：")
display(priority_cases.round(5))

strong_assoc = association.loc[
    association["spearman_rho_with_abs_oof_error"].abs().idxmax()
]

Residual review candidates： 3


,experiment_key,lot_number,wafer_number,residual_anomaly_consensus,residual_driver_feature
0,2024-08-01_04,6,4,2,Stat3_Etch_MV_Heater3Temp
1,2024-08-07_08,8,8,2,Stat3_Etch_MV_Heater4Temp
2,2024-08-07_09,8,9,2,Stat3_Etch_MV_Heater2Temp


Cross-Lot OOD candidates： 5


,experiment_key,lot_number,wafer_number,cross_lot_ood_consensus,ood_driver_feature
0,2024-07-09_01,3,1,3,std__Gas7Flow
1,2024-07-19_01,5,1,2,std__PlatenRFTuningCapacitor
2,2024-08-07_08,8,8,3,late_minus_early__SourceRFPeakToPeak
3,2024-08-07_09,8,9,3,late_minus_early__Heater2Temp
4,2024-08-07_10,8,10,2,late_minus_early__Heater2Temp


Residual + OOD overlap： 2 ['2024-08-07_08', '2024-08-07_09']

Process score vs |OOF error|：


,metric,spearman_rho_with_abs_oof_error,abs_spearman_rho,lot_bootstrap_ci_low,lot_bootstrap_ci_high,bootstrap_valid_reps,n_labeled_wafers,n_lots
0,ood_pca_recon_rmse,0.30948,0.30948,0.00848,0.60309,2000,88,10
1,ood_mahalanobis,0.23648,0.23648,-0.10785,0.59123,2000,88,10
2,residual_mahalanobis,0.11958,0.11958,-0.10355,0.32406,2000,88,10
3,residual_pca_recon_rmse,0.03013,0.03013,-0.18959,0.25073,2000,88,10
4,vm_informed_max_abs_z,0.01040,0.01040,-0.32242,0.34775,2000,88,10



Priority cases：


,case_type,experiment_key,lot_number,wafer_number,residual_anomaly_consensus,cross_lot_ood_consensus,n_flagged_features,n_mr_flagged_features,cv_si_etch_pct,abs_nested_error,vm_informed_max_abs_z,residual_driver_feature,ood_driver_feature,selection_rule
0,A_Process+Model,2024-08-07_10,8,10,1,2,3,5,8.16625,0.57555,1.86279,Stat3_Etch_MV_Heater4Temp,late_minus_early__Heater2Temp,有 process-review evidence 的 labeled wafers 中，|...
1,B_Quality_Contrast,2024-08-01_07,6,7,0,0,0,0,8.94692,0.18857,2.60485,Stat3_Etch_MV_Pressure,robust_width__PlatenRFReflectedPower,Quality CV 位於高端，而且目前 process-review channels 都...


### 這個結果代表什麼？

Residual review candidates 是 **3 / 96**，Pure Cross-Lot OOD candidates 是 **5 / 96**。

真正同時被兩種方法標記的只有 **2 片**：

- `2024-08-07_08`
- `2024-08-07_09`

所以 residual anomaly 和 Cross-Lot OOD 確實不是同一個概念。

再看 process score 和 `|OOF error|` 的 association，最大的是 **`ood_pca_recon_rmse`，Spearman rho = 0.30948**，Lot-cluster bootstrap 95% interval = **[0.00848, 0.60309]**。

這個相關程度只能算部分同步，而且 interval 還很寬。也就是說目前沒有任何一個 anomaly / OOD score 可以完整解釋 VM 為什麼預測錯。

Priority Case A 是 `2024-08-07_10`：它有 OOD、SPC 和最大的 model error evidence。Case B 是 `2024-08-01_07`：Quality CV 很高，但目前 process-review channels 都沒有 flag。這個對照剛好說明 Quality extreme、process unusualness 和 model error 不一定會一起出現。

## 10. Final Findings Table：把整個專案真正能說的結論濃縮成一張表

這張表不是再做一次統計，而是把 02～10 的主要發現收斂成 Final Findings。

每一列都保留四件事：

- Finding
- Supporting evidence
- Correct interpretation
- Limitation

這樣面試時不會只背漂亮數字，也能清楚說明自己知道哪些結論不能下太大。

In [10]:
top_mean = mean_final.loc[mean_final["abs_sequence_adjusted_pearson"].idxmax()]
top_cv = cv_final.loc[cv_final["abs_sequence_adjusted_pearson"].idxmax()]
top_spc_row = spc_feature.sort_values("n_i_violations", ascending=False).iloc[0]
lot8 = lot_summary.loc[lot_summary["lot_number"] == 8].iloc[0]

final_findings = pd.DataFrame([
    {
        "finding": "Wafer quality 有明顯 radial spatial structure",
        "supporting_evidence": (
            f"Outer mean={outer_mean:.3f} µm；inner pooled mean={inner_mean:.3f} µm；"
            f"差={outer_mean-inner_mean:+.3f} µm"
        ),
        "correct_interpretation": "不同 wafer 半徑位置的 etch level 並不均一，Quality analysis 不能只看 wafer-level mean。",
        "limitation": "這是目前 88 片量測 wafer 的 observational spatial summary，不是設備物理因果模型。",
    },
    {
        "finding": "Lot 內 wafer sequence structure 不能忽略",
        "supporting_evidence": (
            f"Mean Si Etch slope<0：{int((lot_sequence['mean_si_etch_slope_per_wafer']<0).sum())}/10 Lots；"
            f"24/24 process features detrending 後 median |Spearman| 下降"
        ),
        "correct_interpretation": "run order 會同時影響 Quality 與 Process pattern，所以後續分析需要 sequence-aware。",
        "limitation": "目前主要移除的是 Lot-specific linear trend，仍可能存在 nonlinear sequence structure。",
    },
    {
        "finding": "扣除線性 sequence trend 後仍留下 Process–Quality candidates",
        "supporting_evidence": (
            f"Mean candidates={len(mean_candidates)}；CV candidates={len(cv_candidates)}；"
            f"Mean 最強={top_mean['sensor']} r={top_mean['sequence_adjusted_pearson']:.3f}"
        ),
        "correct_interpretation": "這些 sensors 可以拿來安排 DOE、equipment log 或 recipe review 的優先順序。",
        "limitation": "目前是 observational association，不是因果證據。",
    },
    {
        "finding": "Phase-I SPC 可以找出 feature-level excursions",
        "supporting_evidence": (
            f"I-chart flags={int(spc_wafer['n_flagged_features'].sum())} events / "
            f"{int((spc_wafer['n_flagged_features']>0).sum())} wafers；"
            f"最多={str(top_spc_row['feature']).replace('Stat3_Etch_MV_','')}"
        ),
        "correct_interpretation": "可以拿來當 wafer / sensor review trigger。",
        "limitation": "目前是 retrospective Phase-I provisional limits，不是 production control limits。",
    },
    {
        "finding": "Process-feature VM 在 nested held-out-Lot evaluation 優於兩個簡單 baselines",
        "supporting_evidence": (
            f"VM RMSE={vm_rmse:.4f}；sequence-only={seq_base:.4f}；"
            f"training-mean={mean_base:.4f} µm"
        ),
        "correct_interpretation": "目前 process engineered features 確實包含對 Mean Si Etch prediction 有用的資訊。",
        "limitation": "還沒有用 sequence-conditional ablation 精確量出 process features 的獨立增量價值。",
    },
    {
        "finding": "不同 observed Lots 的 VM 泛化難度差很大",
        "supporting_evidence": (
            f"Lot 8 RMSE={lot8['vm_rmse']:.4f} µm；"
            f"Lot 1 RMSE={float(lot_summary.loc[lot_summary['lot_number']==1,'vm_rmse'].iloc[0]):.4f} µm"
        ),
        "correct_interpretation": "部署時要監控 group-level performance，不能只看 pooled RMSE。",
        "limitation": "Leave-One-Lot-Out 是 internal validation，不是 future chronological / external production test。",
    },
    {
        "finding": "模型對少數 sensor groups 的依賴特別明顯",
        "supporting_evidence": (
            f"Top permutation sensor={top_perm.iloc[0]['sensor']}；"
            f"mean ΔRMSE={top_perm.iloc[0]['mean_delta_rmse']:.4f} µm"
        ),
        "correct_interpretation": "這些 sensor groups 在目前 held-out-Lot prediction 中提供重要 predictive information。",
        "limitation": "Predictive importance 不等於 causal importance，correlated sensors 也可能共享資訊。",
    },
    {
        "finding": "Residual anomaly 和 Pure Cross-Lot OOD 捕捉不同現象",
        "supporting_evidence": (
            f"Residual candidates={len(res_candidates)}；OOD candidates={len(ood_candidates)}；"
            f"overlap={len(overlap_keys)}"
        ),
        "correct_interpretation": "Residual 比較像 Lot 內 sequence-adjusted unusualness；OOD 比較像相對其他 Lots 的 extrapolation。",
        "limitation": "Residual threshold 是 retrospective；OOD 是 group-aware empirical calibration，不是正式 FPR / conformal guarantee。",
    },
    {
        "finding": "Process unusualness 和 model error 只有部分重疊",
        "supporting_evidence": (
            f"最大 |rho|={abs(strong_assoc['spearman_rho_with_abs_oof_error']):.3f}；"
            f"95% interval=[{strong_assoc['lot_bootstrap_ci_low']:.3f}, "
            f"{strong_assoc['lot_bootstrap_ci_high']:.3f}]"
        ),
        "correct_interpretation": "實際工程判讀要保留 Quality、SPC、OOD、VM error 等多個 evidence channels。",
        "limitation": "目前只有 10 Lots，association 仍屬 exploratory，也沒有 causal identification。",
    },
    {
        "finding": "Lot 8 是目前最值得優先回查的 Lot",
        "supporting_evidence": (
            f"VM RMSE={lot8['vm_rmse']:.4f} µm；OOD={int(lot8['cross_lot_ood_count'])}；"
            f"residual={int(lot8['residual_review_count'])}；SPC-I wafers={int(lot8['spc_i_wafer_count'])}"
        ),
        "correct_interpretation": "多個不同 channel 都在 Lot 8 出現 review signal，所以工程調查可以優先從這個 Lot 開始。",
        "limitation": "多個 signals 並不能相加成 defect probability，也不能直接判定 Lot 8 有製程缺陷。",
    },
])

display(final_findings)

,finding,supporting_evidence,correct_interpretation,limitation
0,Wafer quality 有明顯 radial spatial structure,Outer mean=46.216 µm；inner pooled mean=41.177 ...,不同 wafer 半徑位置的 etch level 並不均一，Quality analysi...,這是目前 88 片量測 wafer 的 observational spatial summ...
1,Lot 內 wafer sequence structure 不能忽略,Mean Si Etch slope<0：10/10 Lots；24/24 process ...,run order 會同時影響 Quality 與 Process pattern，所以後續...,目前主要移除的是 Lot-specific linear trend，仍可能存在 nonli...
2,扣除線性 sequence trend 後仍留下 Process–Quality candi...,Mean candidates=5；CV candidates=7；Mean 最強=Plat...,這些 sensors 可以拿來安排 DOE、equipment log 或 recipe r...,目前是 observational association，不是因果證據。
3,Phase-I SPC 可以找出 feature-level excursions,I-chart flags=46 events / 19 wafers；最多=Gas7Flow,可以拿來當 wafer / sensor review trigger。,目前是 retrospective Phase-I provisional limits，不...
4,Process-feature VM 在 nested held-out-Lot evalu...,VM RMSE=0.1296；sequence-only=0.2209；training-m...,目前 process engineered features 確實包含對 Mean Si E...,還沒有用 sequence-conditional ablation 精確量出 proces...
5,不同 observed Lots 的 VM 泛化難度差很大,Lot 8 RMSE=0.2238 µm；Lot 1 RMSE=0.0652 µm,部署時要監控 group-level performance，不能只看 pooled RMSE。,Leave-One-Lot-Out 是 internal validation，不是 fut...
6,模型對少數 sensor groups 的依賴特別明顯,Top permutation sensor=PlatenRFTuningCapacitor...,這些 sensor groups 在目前 held-out-Lot prediction 中...,Predictive importance 不等於 causal importance，co...
7,Residual anomaly 和 Pure Cross-Lot OOD 捕捉不同現象,Residual candidates=3；OOD candidates=5；overlap=2,Residual 比較像 Lot 內 sequence-adjusted unusualne...,Residual threshold 是 retrospective；OOD 是 group...
8,Process unusualness 和 model error 只有部分重疊,"最大 |rho|=0.309；95% interval=[0.008, 0.603]",實際工程判讀要保留 Quality、SPC、OOD、VM error 等多個 evidenc...,目前只有 10 Lots，association 仍屬 exploratory，也沒有 ca...
9,Lot 8 是目前最值得優先回查的 Lot,VM RMSE=0.2238 µm；OOD=3；residual=2；SPC-I wafers=6,多個不同 channel 都在 Lot 8 出現 review signal，所以工程調查可...,多個 signals 並不能相加成 defect probability，也不能直接判定 L...


### 這個結果代表什麼？

Final Findings table 一共整理出 **10 個主要發現**。

我覺得最重要的不是哪一個數字最大，而是整條分析邏輯可以接起來：

1. Wafer quality 本身有 spatial structure。
2. Quality 和 Process 都有明顯 wafer-sequence structure。
3. 扣掉線性 sequence trend 後，Process–Quality 還是留下一些候選 association。
4. Process features 可以拿來做 Virtual Metrology，而且 nested held-out-Lot 表現比兩個簡單 baseline 好。
5. Explainability、SPC、Residual anomaly、Cross-Lot OOD 和 model error 提供的是不同角度的 evidence，不能硬合成一個總分。

這張表也把 limitation 放在每個 finding 旁邊，所以後續不管是 README、報告或面試，都比較不容易把 exploratory result 講成 causal / production proof。

## 11. Lot-Level Final Summary：最後看哪些 Lots 同時出現多個 review signals

這裡只把不同 channel 並排，不做加權 risk score。

因為：

- VM RMSE 是 prediction difficulty。
- OOD count 是 distribution extrapolation。
- SPC 是 retrospective statistical excursion。
- Quality CV 是 metrology variation。

它們的單位和意義都不同，直接相加反而會失真。

In [11]:
lot_final = lot_summary.sort_values("lot_number").reset_index(drop=True)
display(lot_final.round(5))

lot_high_rmse = lot_final.loc[lot_final["vm_rmse"].idxmax()]
lot_low_rmse = lot_final.loc[lot_final["vm_rmse"].idxmin()]

max_ood_count = int(lot_final["cross_lot_ood_count"].max())
lot_most_ood = (
    lot_final.loc[
        lot_final["cross_lot_ood_count"] == max_ood_count,
        "lot_number",
    ]
    .astype(int)
    .tolist()
)

print(
    "VM RMSE highest：",
    f"Lot {int(lot_high_rmse['lot_number'])} = {lot_high_rmse['vm_rmse']:.5f} µm",
)
print(
    "VM RMSE lowest：",
    f"Lot {int(lot_low_rmse['lot_number'])} = {lot_low_rmse['vm_rmse']:.5f} µm",
)
print(
    "Cross-Lot OOD candidates most：",
    ", ".join(f"Lot {x}" for x in lot_most_ood),
    f"({max_ood_count} wafers)",
)

,lot_number,n_process_wafers,residual_review_count,mean_residual_mahalanobis,cross_lot_ood_count,mean_ood_mahalanobis,spc_i_wafer_count,spc_mr_wafer_count,n_quality_wafers,mean_quality_cv,high_quality_cv_count,vm_rmse,vm_max_abs_error
0,1,10,0,4.20929,0,10.79555,1,2,9,8.07826,0,0.06516,0.11583
1,2,10,0,4.06710,0,9.67794,1,1,10,8.02445,1,0.10996,0.16586
2,3,10,0,4.22266,1,9.67501,2,4,10,8.22720,0,0.14304,0.22756
3,4,10,0,4.32545,0,9.46521,3,1,10,8.21564,1,0.11432,0.21846
4,5,10,0,4.13099,1,10.53817,3,3,10,8.14189,1,0.13771,0.27007
5,6,10,1,4.20019,0,10.09286,1,1,10,8.31099,1,0.11752,0.27963
6,7,6,0,3.67758,0,10.01138,0,1,6,8.25337,0,0.11837,0.20857
7,8,10,2,4.94138,3,12.26297,6,4,10,8.30709,2,0.22384,0.57555
8,9,10,0,3.52332,0,9.54259,0,3,9,8.31790,2,0.06980,0.12378
9,10,10,0,3.96038,0,9.40992,2,3,4,8.31193,1,0.08315,0.10814


VM RMSE highest： Lot 8 = 0.22384 µm
VM RMSE lowest： Lot 1 = 0.06516 µm
Cross-Lot OOD candidates most： Lot 8 (3 wafers)


### 這個結果代表什麼？

Lot-level 表格裡，**Lot 8 的 VM RMSE 最高，是 0.22384 µm**。

同一個 Lot 8 還有：

- **2 片** residual review candidates
- **3 片** Cross-Lot OOD candidates
- **6 片** wafers 有 SPC I-chart flag
- **4 片** wafers 有 MR flag
- 最大 `|OOF error|` = **0.57555 µm**

所以 Lot 8 的確是目前最值得優先回查的 Lot。

但這句話的意思是「值得優先調查」，不是「已經證明 Lot 8 有 defect」。這些 channel 彼此不是統計獨立，而且每個指標的定義也不同，所以不能把它們加成一個 defect probability。

## 12. 目前距離 Production 還差什麼？

這個專案現在已經是一個完整的 **analysis / modeling prototype**，但還不能叫 production-ready。

主要還差：

1. **更多 production data。** 現在只有 10 Lots，無法代表不同 tool、recipe、fab、maintenance state 或長時間 drift。
2. **Chronological / external validation。** Leave-One-Lot-Out 可以測目前資料中的 unseen-Lot generalization，但不是未來時間點的真正 deployment test。
3. **Sequence-conditional ablation。** VM 比 sequence-only baseline 好，但還要另外設計 nested ablation，才能精確回答 process features 在控制 sequence 後增加多少 prediction value。
4. **正式 SPC baseline。** 現在是 retrospective Phase-I limits，需要用獨立、穩定期 production data 建 Phase-II monitoring limits。
5. **OOD calibration validation。** 10 已經改成 outer-training-only fixed feature universe + Inner LOGO cross-fitted empirical calibration，但仍沒有 exact conformal coverage 或正式 false-positive-rate guarantee。
6. **DOE / engineering validation。** Process–Quality correlation 和 explainability 只能提供 hypothesis priority，不能代替 intervention。
7. **Monitoring / retraining policy。** 正式上線還要定義 model drift、sensor drift、missing data、OOD、retraining trigger 和 rollback。

## 13. 輸出 Final Summary Artifacts

最後把 11 真正會給 README、報告或 dashboard 使用的 tables 存成 CSV。

寫出後會再讀回來核對 row count 和欄位，避免最後輸出的檔案和 Notebook 畫面不是同一版。

In [12]:
outputs = {
    PROCESSED_DATA_DIR / "11_headline_summary.csv": headline,
    PROCESSED_DATA_DIR / "11_quality_sequence_summary.csv": quality_sequence_summary,
    PROCESSED_DATA_DIR / "11_process_quality_candidates.csv": process_quality_summary,
    PROCESSED_DATA_DIR / "11_priority_cases.csv": priority_cases,
    PROCESSED_DATA_DIR / "11_final_findings.csv": final_findings,
    PROCESSED_DATA_DIR / "11_lot_summary.csv": lot_final,
}

validation = []

for path, df in outputs.items():
    df.to_csv(path, index=False)

    check = pd.read_csv(path)

    if len(check) != len(df):
        raise ValueError(f"{path.name} row count validation failed")

    if list(check.columns) != list(df.columns):
        raise ValueError(f"{path.name} schema validation failed")

    validation.append({
        "artifact": path.name,
        "rows": len(check),
        "schema_match": True,
    })

validation_df = pd.DataFrame(validation)
display(validation_df)

print("11 artifact validation：PASS")

,artifact,rows,schema_match
0,11_headline_summary.csv,14,True
1,11_quality_sequence_summary.csv,8,True
2,11_process_quality_candidates.csv,12,True
3,11_priority_cases.csv,2,True
4,11_final_findings.csv,10,True
5,11_lot_summary.csv,10,True


11 artifact validation：PASS


### 這個結果代表什麼？

最後成功輸出 **6 個 Final Summary artifacts**，而且全部重新讀回後都是 `schema_match=True`。

這些檔案分別保存：

- Headline metrics
- Quality / sequence summary
- Process–Quality candidates
- Priority cases
- Final findings
- Lot-level summary

所以之後 README、報告或 dashboard 如果需要引用最終結果，可以直接使用這 6 個檔案，不需要再從前面的 notebook 手動抄數字。

## 14. 整個專案的最後結論

把 00～10 串起來後，我會把這個專案定位成一個 **Semiconductor Etch Process Intelligence Prototype**。

### 1. 我先確認資料本身不是單純 IID samples

Quality 和 Process 都有很明顯的 Lot / wafer-sequence structure。Mean Si Etch 在 **10 / 10 Lots** 都隨 wafer order 下降，而 24 個 process features 做 Lot-specific linear detrending 後，median `|within-Lot Spearman|` 從 **0.388** 降到 **0.085**。

所以後面的 correlation、SPC 和 anomaly analysis 都不能忽略 run order。

### 2. Wafer quality 還有明顯 spatial structure

平均 Mean Si Etch 是 **43.668 µm**，而 `>=75 mm` 外圈的 pooled mean 比內部高約 **5.039 µm**。

這表示只看一個 wafer-level average 會漏掉 spatial non-uniformity。

### 3. 扣掉線性 sequence trend 後，Process–Quality association 並沒有全部消失

Mean Si Etch 有 **5 個** exploratory candidates，最明顯的是 `PlatenRFLoadPower`，r = **-0.609**；CV 有 **7 個** candidates，最明顯的是 `Heater4Temp`，r = **+0.551**。

這些結果比較適合拿來排 DOE / equipment review priority，而不是直接當 causal proof。

### 4. Process features 可以支援 Virtual Metrology

Nested Leave-One-Lot-Out OOF：

- RMSE = **0.12963 µm**
- MAE = **0.09723 µm**
- R² = **0.89477**

而 Sequence-only baseline RMSE 是 **0.22093 µm**，Training-Mean baseline 是 **0.40437 µm**。

所以目前 process-feature model 在相同 nested evaluation 裡確實比較好，但還需要 sequence-conditional ablation 才能精確量出獨立增量價值。

### 5. 模型對少數 sensor groups 的依賴很明顯

Grouped permutation 第一名是 **`PlatenRFTuningCapacitor`**，平均 ΔRMSE = **+0.24996 µm**。Final ElasticNet 的 coefficient aggregation 第一名也是同一個 sensor group。

這代表模型反覆使用這組資訊，但仍然只是 predictive dependence，不是製程因果。

### 6. SPC、Residual Anomaly、Cross-Lot OOD 和 Model Error 是不同的訊號

Phase-I SPC 找到 **46 個 I-chart feature-wafer flags**；Residual review 有 **3 / 96**；Pure Cross-Lot OOD 有 **5 / 96**，兩者只有 **2 片**重疊。

Process score 對 `|OOF error|` 最大的 association 是 `ood_pca_recon_rmse`，rho = **0.309**，95% interval = **[0.008, 0.603]**。

所以沒有一個 anomaly score 可以代替所有工程判讀。

### 7. Lot 8 是目前最值得優先回查的 Lot

Lot 8 同時有：

- 最高 VM RMSE：**0.22384 µm**
- **3 片** OOD candidates
- **2 片** residual candidates
- **6 片** SPC I-chart flagged wafers
- 最大 `|OOF error|`：**0.57555 µm**

所以後續如果要做 raw trace、equipment event、maintenance log 或 recipe context review，我會先從 Lot 8 開始。

但目前只能說「優先調查」，不能直接說它有 defect。

---

整體來說，這個專案已經完成一條完整流程：

**Raw / Quality Understanding → Sequence-aware EDA → Process–Quality Association → Phase-I SPC → Multivariate Analysis → Feature Engineering → Nested Virtual Metrology → Explainability → Residual Anomaly / Pure Cross-Lot OOD → Final Findings**

我認為這個版本最重要的地方，不只是模型成績，而是每一層 evidence 都有保留自己的定義和限制，沒有把 correlation、importance、SPC、OOD 或 model error 混成同一件事。